# ENEXRE - Full RE Training di Google Colab

Notebook ini menjalankan training Relation Extraction BC5CDR menggunakan PubMedBERT dari repository `enexre`.

Sebelum menjalankan notebook:

1. Pilih `Runtime > Change runtime type > GPU`.
2. Push repository `enexre` ke GitHub.
3. Pastikan file `data/bc5cdr/*.txt` tersedia, atau file processed RE sudah tersedia di `data/processed/re/`.

Output utama akan disimpan ke `results/re/`, `logs/re/`, `predictions/re/`, dan `checkpoints/re/`.

## 1. Cek GPU

In [1]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Fri Jul 31 15:27:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository dari GitHub

Ganti `GITHUB_REPO_URL` dengan URL repository kamu, misalnya `https://github.com/USERNAME/enexre.git`.

Untuk repository private, gunakan format `https://TOKEN@github.com/USERNAME/enexre.git`.

In [2]:
#@title Konfigurasi repository
GITHUB_REPO_URL = "https://github.com/defrein/enexre.git"  #@param {type:"string"}
PROJECT_DIR = "/content/enexre"  #@param {type:"string"}

import os
import shutil
from pathlib import Path

os.chdir("/content")

project_path = Path(PROJECT_DIR)

if project_path.exists():
    print(f"Project directory already exists: {project_path}")
else:
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}

os.chdir(project_path)
print("Current directory:", Path.cwd())
!git status --short

Cloning into '/content/enexre'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 97 (delta 39), reused 81 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 7.42 MiB | 11.99 MiB/s, done.
Resolving deltas: 100% (39/39), done.
Current directory: /content/enexre


## 3. Install Dependency

In [3]:
!pip install -q -r requirements.txt

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.5/914.5 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into acc

## 4. Cek Dataset RE

Cell ini mengecek apakah dataset processed RE sudah tersedia. Jika belum tersedia, jalankan preprocessing pada cell berikutnya dari file BC5CDR mentah.

In [5]:
from pathlib import Path

required_re_files = [
    Path("data/processed/re/train.jsonl"),
    Path("data/processed/re/dev.jsonl"),
    Path("data/processed/re/test.jsonl"),
]

for path in required_re_files:
    print(path, "OK" if path.exists() else "MISSING")

all_processed_files_exist = all(path.exists() for path in required_re_files)
print("Processed RE dataset ready:", all_processed_files_exist)

data/processed/re/train.jsonl OK
data/processed/re/dev.jsonl OK
data/processed/re/test.jsonl OK
Processed RE dataset ready: True


## 5. Opsional: Validasi dan Preprocessing RE

Jalankan cell ini jika `data/processed/re/*.jsonl` belum tersedia tetapi file BC5CDR mentah sudah ada di:

- `data/bc5cdr/train.txt`
- `data/bc5cdr/dev.txt`
- `data/bc5cdr/test.txt`

In [7]:
RUN_RE_PREPROCESSING = True  #@param {type:"boolean"}

if RUN_RE_PREPROCESSING:
    !python scripts/validate_bc5cdr.py
    !python scripts/build_re_dataset.py
else:
    print("Skip RE preprocessing. Set RUN_RE_PREPROCESSING=True if processed RE files are missing.")

Validation report written to results/dataset_validation.json
Passed: True
train: docs=500, Chemical=5203, Disease=4182, CID=1038, invalid_ann=0, invalid_rel=0, duplicate_rel=0
dev: docs=500, Chemical=5347, Disease=4244, CID=1012, invalid_ann=0, invalid_rel=0, duplicate_rel=0
test: docs=500, Chemical=5385, Disease=4424, CID=1066, invalid_ann=0, invalid_rel=0, duplicate_rel=0
duplicate_pmids_across_subsets=0
config.json: 100% 385/385 [00:00<00:00, 2.04MB/s]
tokenizer_config.json: 100% 28.0/28.0 [00:00<00:00, 154kB/s]
vocab.txt: 100% 226k/226k [00:00<00:00, 55.1MB/s]
Building RE train: 100% 500/500 [00:06<00:00, 73.87it/s] 
Building RE dev: 100% 500/500 [00:05<00:00, 93.19it/s] 
Building RE test: 100% 500/500 [00:06<00:00, 71.95it/s] 
RE dataset written to data/processed/re
Report written to results/re_preprocessing_report.json
Passed: True
train: docs=500, CID=1038, non-CID=4394, total=5432, over_max_len=334, baseline_f1=0.3209
dev: docs=500, CID=1012, non-CID=4249, total=5261, over_max_

## 6. Lihat Ringkasan Dataset dan Baseline

Cell ini membaca `results/re_preprocessing_report.json` untuk melihat jumlah kandidat dan baseline co-occurrence.

In [8]:
import json
from pathlib import Path

report_path = Path("results/re_preprocessing_report.json")
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    for split, values in report["summary"]["subsets"].items():
        baseline = values["baseline_cooccurrence"]
        print(
            f"{split}: CID={values['cid_pairs']} non-CID={values['non_cid_pairs']} "
            f"total={values['candidate_pairs']} baseline_f1={baseline['f1']:.4f} "
            f"over_512={values['marked_inputs_over_max_length']}"
        )
else:
    print("Missing report:", report_path)

train: CID=1038 non-CID=4394 total=5432 baseline_f1=0.3209 over_512=334
dev: CID=1012 non-CID=4249 total=5261 baseline_f1=0.3227 over_512=244
test: CID=1066 non-CID=4339 total=5405 baseline_f1=0.3295 over_512=274


## 7. Smoke Test Training RE

Smoke test memastikan model, tokenizer marker, cropping, loss, threshold selection, evaluasi, dan penyimpanan metrik berjalan di Colab.

In [7]:
RUN_SMOKE_TEST = True  #@param {type:"boolean"}

if RUN_SMOKE_TEST:
    !python scripts/train_re.py --smoke-test --cleanup-smoke-checkpoint
else:
    print("Skip smoke test.")

config.json: 100% 385/385 [00:00<00:00, 2.52MB/s]
tokenizer_config.json: 100% 28.0/28.0 [00:00<00:00, 137kB/s]
vocab.txt: 100% 226k/226k [00:00<00:00, 51.9MB/s]

pytorch_model.bin: downloading bytes:  61% 270M/440M [00:02<00:00, 207MB/s, 23.4MB/s  ]  
pytorch_model.bin: downloading bytes:  95% 417M/440M [00:03<00:00, 182MB/s, 35.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 418M/418M [00:07<00:00, 56.9MB/s, 35.8MB/s  ] ]
pytorch_model.bin: reconstructing file: 100% 440M/440M [00:07<00:00, 60.0MB/s, 31.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 31390.24it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 


## 8. Opsional: Seleksi Learning Rate RE

Gunakan cell ini untuk memilih konfigurasi awal berdasarkan F1 kelas CID pada development set. Test set belum digunakan pada tahap ini.

In [9]:
RUN_LR_SELECTION = True  #@param {type:"boolean"}
LR_SELECTION_SEED = 13  #@param {type:"integer"}
LR_SELECTION_BATCH_SIZE = 8  #@param [8, 16] {type:"raw"}
LR_SELECTION_EPOCHS = 10  #@param {type:"integer"}
LR_SELECTION_PATIENCE = 2  #@param {type:"integer"}

import os

if RUN_LR_SELECTION:
    for lr in ["1e-5", "3e-5", "5e-5"]:
        run_name = f"re_seed{LR_SELECTION_SEED}_lr{lr}_bs{LR_SELECTION_BATCH_SIZE}"
        command = (
            f"python scripts/train_re.py "
            f"--seed {LR_SELECTION_SEED} "
            f"--learning-rate {lr} "
            f"--batch-size {LR_SELECTION_BATCH_SIZE} "
            f"--epochs {LR_SELECTION_EPOCHS} "
            f"--patience {LR_SELECTION_PATIENCE} "
            f"--run-name {run_name}"
        )
        print("Running:", command)
        exit_code = os.system(command)
        if exit_code != 0:
            raise RuntimeError(f"RE training failed for {run_name}")
else:
    print("Skip learning-rate selection.")

Running: python scripts/train_re.py --seed 13 --learning-rate 1e-5 --batch-size 8 --epochs 10 --patience 2 --run-name re_seed13_lr1e-5_bs8
Running: python scripts/train_re.py --seed 13 --learning-rate 3e-5 --batch-size 8 --epochs 10 --patience 2 --run-name re_seed13_lr3e-5_bs8
Running: python scripts/train_re.py --seed 13 --learning-rate 5e-5 --batch-size 8 --epochs 10 --patience 2 --run-name re_seed13_lr5e-5_bs8


RuntimeError: RE training failed for re_seed13_lr5e-5_bs8

In [17]:
!ps aux | grep train_re.py

root       46733  0.0  0.0   7376  3484 ?        S    17:53   0:00 /bin/bash -c ps aux | grep train_re.py
root       46735  0.0  0.0   6616  2416 ?        S    17:53   0:00 grep train_re.py


In [ ]:
!python scripts/train_re.py --seed 13 --learning-rate 5e-5 --batch-size 8 --epochs 10 --patience 2 --run-name re_seed13_lr5e-5_bs8

Loading weights: 100% 199/199 [00:00<00:00, 19080.27it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading 

In [15]:
from google.colab import drive
from pathlib import Path
import shutil, time

drive.mount("/content/drive")

stamp = time.strftime("%Y%m%d_%H%M%S")
backup_dir = Path(f"/content/drive/MyDrive/enexre_outputs/re_backup_{stamp}")
backup_dir.mkdir(parents=True, exist_ok=True)

for source in [
    Path("results/re"),
    Path("logs/re"),
    Path("predictions/re"),
    Path("checkpoints/re"),
    Path("results/re_preprocessing_report.json"),
    Path("data/processed/re"),
]:
    if source.exists():
        target = backup_dir / source.name
        if source.is_dir():
            shutil.copytree(source, target, dirs_exist_ok=True)
        else:
            shutil.copy2(source, target)
        print("saved:", source, "->", target)
    else:
        print("missing:", source)

print("Backup dir:", backup_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved: results/re -> /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613/re
saved: logs/re -> /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613/re
saved: predictions/re -> /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613/re
saved: checkpoints/re -> /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613/re
missing: results/re_preprocessing_report.json
saved: data/processed/re -> /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613/re
Backup dir: /content/drive/MyDrive/enexre_outputs/re_backup_20260730_174613


In [16]:
from pathlib import Path
import shutil, time

backup_dir = Path("/content/drive/MyDrive/enexre_outputs/re_backup_FINAL")
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    (Path("results/re"), "results_re"),
    (Path("logs/re"), "logs_re"),
    (Path("predictions/re"), "predictions_re"),
    (Path("checkpoints/re"), "checkpoints_re"),
    (Path("results/re_preprocessing_report.json"), "re_preprocessing_report.json"),
    (Path("data/processed/re"), "processed_re"),
]

for source, target_name in items:
    target = backup_dir / target_name
    if source.exists():
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        if source.is_dir():
            shutil.copytree(source, target)
        else:
            shutil.copy2(source, target)
        print("saved:", source, "->", target)
    else:
        print("missing:", source)

print("Final backup dir:", backup_dir)

saved: results/re -> /content/drive/MyDrive/enexre_outputs/re_backup_FINAL/results_re
saved: logs/re -> /content/drive/MyDrive/enexre_outputs/re_backup_FINAL/logs_re
saved: predictions/re -> /content/drive/MyDrive/enexre_outputs/re_backup_FINAL/predictions_re
saved: checkpoints/re -> /content/drive/MyDrive/enexre_outputs/re_backup_FINAL/checkpoints_re
missing: results/re_preprocessing_report.json
saved: data/processed/re -> /content/drive/MyDrive/enexre_outputs/re_backup_FINAL/processed_re
Final backup dir: /content/drive/MyDrive/enexre_outputs/re_backup_FINAL


## 9. Full Training RE Tiga Random Seed

Gunakan cell ini setelah konfigurasi final dipilih dari development set. Default awal memakai learning rate `1e-5`, batch size 8, dan tiga seed `13,42,100`.

In [ ]:
RUN_FINAL_THREE_SEED_TRAINING = True  #@param {type:"boolean"}
FINAL_SEEDS = "13"  #@param {type:"string": 32, 42, 100}
LEARNING_RATE = "3e-5"  #@param ["1e-5", "3e-5", "5e-5"] {allow-input: true}
BATCH_SIZE = 8  #@param [8, 16] {type:"raw"}
EPOCHS = 10  #@param {type:"integer"}
PATIENCE = 2  #@param {type:"integer"}

if RUN_FINAL_THREE_SEED_TRAINING:
    seeds = [int(seed.strip()) for seed in FINAL_SEEDS.split(",") if seed.strip()]
    for seed in seeds:
        run_name = f"final_re_seed{seed}_lr{LEARNING_RATE}_bs{BATCH_SIZE}"
        command = (
            f"python scripts/train_re.py "
            f"--seed {seed} "
            f"--learning-rate {LEARNING_RATE} "
            f"--batch-size {BATCH_SIZE} "
            f"--epochs {EPOCHS} "
            f"--patience {PATIENCE} "
            f"--run-name {run_name}"
        )
        print("Running:", command)
        exit_code = os.system(command)
        if exit_code != 0:
            raise RuntimeError(f"RE training failed for {run_name}")
else:
    print("Skip final three-seed RE training.")

Running: python scripts/train_re.py --seed 13 --learning-rate 1e-5 --batch-size 8 --epochs 10 --patience 2 --run-name final_re_seed13_lr1e-5_bs8


RuntimeError: RE training failed for final_re_seed13_lr1e-5_bs8

## 10. Ringkas Hasil Training RE

Cell ini membaca semua file `results/re/*_metrics.json` dan mengurutkan hasil berdasarkan `best_dev_f1`.

In [ ]:
import json
from pathlib import Path

metrics_rows = []
for path in sorted(Path("results/re").glob("*_metrics.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    metrics_rows.append({
        "file": str(path),
        "run_name": data.get("run_name"),
        "seed": data.get("seed"),
        "learning_rate": data.get("learning_rate"),
        "batch_size": data.get("batch_size"),
        "best_epoch": data.get("best_epoch"),
        "best_dev_f1": data.get("best_dev_f1"),
        "best_threshold": data.get("best_threshold"),
        "checkpoint_dir": data.get("checkpoint_dir"),
        "smoke_test": data.get("smoke_test"),
    })

metrics_rows = sorted(metrics_rows, key=lambda row: row.get("best_dev_f1") or -1, reverse=True)

for row in metrics_rows:
    print(
        f"{row['best_dev_f1']:.4f} | {row['run_name']} | "
        f"seed={row['seed']} lr={row['learning_rate']} bs={row['batch_size']} "
        f"best_epoch={row['best_epoch']} threshold={row['best_threshold']} "
        f"smoke={row['smoke_test']}"
    )

## 11. Evaluasi RE Tiga Seed pada Test Set

Cell ini mengevaluasi checkpoint `final_re_seed...` pada `data/processed/re/test.jsonl`. Threshold yang digunakan adalah threshold terbaik dari development set masing-masing run.

In [ ]:
RUN_THREE_SEED_TEST_EVALUATION = True  #@param {type:"boolean"}
TEST_BATCH_SIZE = 16  #@param [8, 16, 32] {type:"raw"}
FINAL_RUN_PREFIX = "final_re_seed"  #@param {type:"string"}

import json
import os
import statistics
from pathlib import Path

if RUN_THREE_SEED_TEST_EVALUATION:
    metric_paths = sorted(Path("results/re").glob(f"{FINAL_RUN_PREFIX}*_metrics.json"))
    if not metric_paths:
        raise FileNotFoundError(f"No RE metrics found for prefix: {FINAL_RUN_PREFIX}")

    test_rows = []
    for metric_path in metric_paths:
        train_metrics = json.loads(metric_path.read_text(encoding="utf-8"))
        if train_metrics.get("smoke_test"):
            continue
        run_name = train_metrics["run_name"]
        checkpoint_dir = train_metrics["checkpoint_dir"]
        threshold = train_metrics.get("best_threshold", 0.5)
        output_path = f"results/re/{run_name}_test_metrics.json"
        predictions_path = f"predictions/re/{run_name}_test_predictions.jsonl"
        command = (
            f"python scripts/evaluate_re.py "
            f"--checkpoint {checkpoint_dir} "
            f"--threshold {threshold} "
            f"--batch-size {TEST_BATCH_SIZE} "
            f"--output {output_path} "
            f"--predictions {predictions_path}"
        )
        print("Running:", command)
        exit_code = os.system(command)
        if exit_code != 0:
            raise RuntimeError(f"RE test evaluation failed for {run_name}")

        test_metrics = json.loads(Path(output_path).read_text(encoding="utf-8"))
        test = test_metrics["test"]
        test_rows.append({
            "run_name": run_name,
            "seed": train_metrics.get("seed"),
            "checkpoint_dir": checkpoint_dir,
            "best_dev_f1": train_metrics.get("best_dev_f1"),
            "threshold": threshold,
            "test_precision": test["precision"],
            "test_recall": test["recall"],
            "test_f1": test["f1"],
            "true_positive": test["true_positive"],
            "false_positive": test["false_positive"],
            "false_negative": test["false_negative"],
            "metrics_path": output_path,
            "predictions_path": predictions_path,
        })

    if not test_rows:
        raise RuntimeError("No non-smoke final RE runs were evaluated.")

    summary = {"runs": test_rows, "aggregate": {}}
    for key in ["best_dev_f1", "test_precision", "test_recall", "test_f1", "false_positive"]:
        values = [row[key] for row in test_rows]
        summary["aggregate"][key] = {
            "mean": statistics.mean(values),
            "std": statistics.stdev(values) if len(values) > 1 else 0.0,
        }

    summary_path = Path("results/re/final_three_seed_test_summary.json")
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    for row in test_rows:
        print(
            f"{row['run_name']} | threshold={row['threshold']} "
            f"test_f1={row['test_f1']:.4f} precision={row['test_precision']:.4f} recall={row['test_recall']:.4f} "
            f"FP={row['false_positive']}"
        )
    print("\nFinal three-seed RE test summary:")
    for key, values in summary["aggregate"].items():
        print(f"{key}: mean={values['mean']:.4f} std={values['std']:.4f}")
    print("Summary:", summary_path)
else:
    print("Skip three-seed RE test evaluation.")

## 12. Bandingkan dengan Baseline Co-occurrence

Cell ini menampilkan baseline test dari preprocessing dan hasil RE final jika sudah dievaluasi.

Jika `results/re/final_three_seed_test_summary.json` belum tersedia, cell memakai hasil single best-test dari `results/re/best_test_metrics.json`.

In [4]:
import json
from pathlib import Path

baseline_report = json.loads(Path("results/re_preprocessing_report.json").read_text(encoding="utf-8"))
baseline = baseline_report["summary"]["subsets"]["test"]["baseline_cooccurrence"]
print("Baseline co-occurrence test:")
print(f"precision={baseline['precision']:.4f} recall={baseline['recall']:.4f} f1={baseline['f1']:.4f} FP={baseline['false_positive']}")

summary_path = Path("results/re/final_three_seed_test_summary.json")
best_test_path = Path("results/re/best_test_metrics.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    agg = summary["aggregate"]
    delta_f1 = agg["test_f1"]["mean"] - baseline["f1"]
    fp_reduction = (baseline["false_positive"] - agg["false_positive"]["mean"]) / baseline["false_positive"] * 100
    print("\nPubMedBERT RE mean test:")
    print(f"precision={agg['test_precision']['mean']:.4f} recall={agg['test_recall']['mean']:.4f} f1={agg['test_f1']['mean']:.4f}")
    print(f"Delta F1={delta_f1:.4f}")
    print(f"False positive reduction={fp_reduction:.2f}%")
elif best_test_path.exists():
    result = json.loads(best_test_path.read_text(encoding="utf-8"))
    test = result["test"]
    selected_run = result.get("selected_run") or {}
    delta_f1 = test["f1"] - baseline["f1"]
    fp_reduction = (baseline["false_positive"] - test["false_positive"]) / baseline["false_positive"] * 100
    print("\nPubMedBERT RE final test:")
    print(f"run={selected_run.get('run_name', result.get('checkpoint'))} threshold={test['threshold']:.2f}")
    print(f"precision={test['precision']:.4f} recall={test['recall']:.4f} f1={test['f1']:.4f} FP={test['false_positive']}")
    print(f"Delta F1={delta_f1:.4f}")
    print(f"False positive reduction={fp_reduction:.2f}%")
else:
    print("\nFinal RE summary belum tersedia.")

Baseline co-occurrence test:
precision=0.1972 recall=1.0000 f1=0.3295 FP=4339

PubMedBERT RE final test:
run=re_seed13_lr3e-5_bs8 threshold=0.70
precision=0.6694 recall=0.6914 f1=0.6802 FP=364
Delta F1=0.3507
False positive reduction=91.61%


## 13. Simpan Output ke Google Drive

Simpan hasil training agar tidak hilang saat runtime Colab berhenti.

In [ ]:
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/enexre_outputs"  #@param {type:"string"}

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    output_dir = Path(DRIVE_OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for source, target_name in [
        (Path("results/re"), "results_re"),
        (Path("logs/re"), "logs_re"),
        (Path("predictions/re"), "predictions_re"),
        (Path("checkpoints/re"), "checkpoints_re"),
        (Path("data/processed/re"), "processed_re"),
        (Path("results/re_preprocessing_report.json"), "re_preprocessing_report.json"),
    ]:
        target = output_dir / target_name
        if source.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(source, target)
            print(f"Copied {source} -> {target}")
        elif source.is_file():
            shutil.copy2(source, target)
            print(f"Copied {source} -> {target}")
        else:
            print(f"Skip missing source: {source}")
else:
    print("Skip saving to Drive.")

## 14. Catatan

- Jangan push `checkpoints/re/` ke GitHub biasa kecuali memakai Git LFS.
- Untuk laporan eksperimen, catat seed, learning rate, batch size, `best_dev_f1`, threshold terbaik, dan hasil test.
- Test set hanya digunakan setelah konfigurasi dan threshold dipilih dari development set.
- Setelah RE gold entities selesai, lanjutkan ke pengujian pipeline NER-RE.